# Calibrated Popularity and history-aware calibration

This notebook applies fixed Calibrated Popularity (CP) and history-aware CP to ALS recommendations. Settings are selected using a validation split made only from training data; only selected settings are then evaluated on the held-out test data. The lambda-zero baseline reproduction assertion is retained.

**Inputs:** exported `top-100-candidates` and `top-10-baseline` Parquet tables from `als-baseline.ipynb`, plus `data/ml-1m/ratings.dat`. Validation candidates are generated by rank-50 ALS trained only on a seed-7 reduced training subset; test recommendations use the original full-training baseline exports.

**Method:** for each user, build the top-10 greedily using `(1 - lambda_u) * Rel(L) - lambda_u * JS(p(u), q(L))`. CP uses `lambda_u=lambda`; history-aware CP uses `lambda_u=lambda * min(1, n_u/N0)`.

**Selection:** choose each method's highest validation NDCG@10 setting under a configurable validation UPD ceiling, and separately select a history-aware setting using the matched-UPD interpolation from `analysis/history_group_analysis.py`. Test metrics are only reported in `selected_on_validation.csv` and are never used for selection.

Run the cells in order with Scala 2.12.18 / Java 17. Spark candidate/profile work stays on DataFrames; the existing small driver-side `Calib` arrays are reused unchanged.

In [1]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`
import $ivy.`org.apache.spark::spark-mllib:3.5.9`

import org.apache.spark.sql.{DataFrame, SparkSession}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types._

import $ivy.$
import $ivy.$
import org.apache.spark.sql.{DataFrame, SparkSession}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types._

In [2]:
val spark = SparkSession.builder()
  .appName("CS 7265 - Calibrated Popularity")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._

assert(spark.version == "3.5.9")
println(s"Spark ${spark.version}, Scala ${scala.util.Properties.versionNumberString}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/15 20:57:08 WARN Utils: Your hostname, Jrdns-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.194 instead (on interface en0)
26/11/15 20:57:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/11/15 20:57:09 INFO SparkContext: Running Spark version 3.5.9
26/11/15 20:57:09 INFO SparkContext: OS info Mac OS X, 26.5.1, aarch64
26/11/15 20:57:09 INFO SparkContext: Java version 17.0.20.1
26/11/15 20:57:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/15 20:57:09 INFO ResourceUtils: ==============================================================
26/11/15 20:57:09 INFO ResourceUtils: No custom resources configured for spark.driver.
26/11/15 20:57:09 INFO ResourceUtils: ==============================================================
26/11/15 20:57:09 INFO SparkContext: Submitt

Spark 3.5.9, Scala 2.12.18


spark: SparkSession = org.apache.spark.sql.SparkSession@157505e4
import spark.implicits._

## 1. Configuration

Set `splitSeed`, `baselineRunId`, and `profileRunId` to matching runs before execution. Use seeds 42, 43, and 44 for the three split repetitions; the validation-only split remains seed 7. `validationUpdMax` controls the validation-only UPD constraint. Since validation ALS uses 80% of the outer training rows, history cutoffs are proportionally reduced (50/100/200 become 40/80/160) for validation; full-training test cutoffs remain unchanged.

In [3]:
val baselineRunId = "2026-11-16T01-44-41.706431Z"   // e.g. 2026-11-02T18-04-11.123456Z
val profileRunId = "2026-11-16T01-50-34.019005Z"   // timestamped run under outputs/popularity-profiles/
val splitSeed = 42L
val alsSeed = splitSeed
require(Set(42L, 43L, 44L).contains(splitSeed), s"Unsupported split seed: $splitSeed")
val trainingFraction = 0.8
val listSize = 10
val candidatePoolSize = 100
val validationSplitSeed = 7L
val validationTrainingFraction = 0.8
val validationUpdMax = 0.35

val lambdas = Seq(0.1, 0.3, 0.5, 0.7, 0.9)
val n0Values = Seq(50, 100, 200)

val shortCutoff = 50     // short:  n_u < 50
val longCutoff = 200     // medium: 50 <= n_u <= 200, long: n_u > 200
val validationShortCutoff = math.round(shortCutoff * validationTrainingFraction).toInt
val validationLongCutoff = math.round(longCutoff * validationTrainingFraction).toInt

// Optional historical regression check; applies only to this baseline run.
val historicalRegressionBaselineRunId = "2026-11-16T01-44-41.706431Z"
val checkHistoricalBaselineRegression = false
val historicalPrecision = 0.0567
val historicalRecall = 0.0391
val historicalNdcg = 0.0513

val ratingsPath = Seq("data/ml-1m/ratings.dat", "../data/ml-1m/ratings.dat")
  .find(p => new java.io.File(p).isFile)
  .getOrElse(throw new IllegalArgumentException("Run from the repository root or notebooks folder with data/ml-1m/ present."))
val repositoryRoot = new java.io.File(ratingsPath).getCanonicalFile.getParentFile.getParentFile.getParentFile
val baselineRunDir = new java.io.File(repositoryRoot, s"outputs/als-baseline/$baselineRunId")
require(baselineRunDir.isDirectory, s"Baseline run folder not found: ${baselineRunDir.getPath}")
require(profileRunId.nonEmpty, "Set profileRunId to the matching outputs/popularity-profiles run.")
val profileRunDir = new java.io.File(repositoryRoot, s"outputs/popularity-profiles/$profileRunId")
require(profileRunDir.isDirectory, s"Popularity profile run folder not found: ${profileRunDir.getPath}")

val calibrationRunId = java.time.Instant.now().toString.replace(":", "-")
val calibrationDir = new java.io.File(repositoryRoot, s"outputs/calibration/$calibrationRunId")
println(s"Baseline run:    ${baselineRunDir.getPath}")
println(s"Calibration run: ${calibrationDir.getPath}")

Baseline run:    /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/als-baseline/2026-11-16T01-44-41.706431Z
Calibration run: /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/calibration/2026-11-16T01-57-11.331215Z


baselineRunId: String = "2026-11-16T01-44-41.706431Z"
profileRunId: String = "2026-11-16T01-50-34.019005Z"
splitSeed: Long = 42L
alsSeed: Long = 42L
trainingFraction: Double = 0.8
listSize: Int = 10
candidatePoolSize: Int = 100
validationSplitSeed: Long = 7L
validationTrainingFraction: Double = 0.8
validationUpdMax: Double = 0.35
lambdas: Seq[Double] = List(0.1, 0.3, 0.5, 0.7, 0.9)
n0Values: Seq[Int] = List(50, 100, 200)
shortCutoff: Int = 50
longCutoff: Int = 200
validationShortCutoff: Int = 40
validationLongCutoff: Int = 160
historicalRegressionBaselineRunId: String = "2026-11-16T01-44-41.706431Z"
checkHistoricalBaselineRegression: Boolean = false
historicalPrecision: Double = 0.0567
historicalRecall: Double = 0.0391
historicalNdcg: Double = 0.0513
ratingsPath: String = "../data/ml-1m/ratings.dat"
repositoryRoot: java.io.File = /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration
baselineRunDir: java.io.File = /Users/jrdn/Documents/GitHub/history-aware-popularity-calibra

## 2. Rebuild the outer training/test split

This repeats the baseline's per-user split: `xxhash64(splitSeed, userId, movieId)` ordering, `floor(0.8 * n)` training rows per user. IDs remain `Int` so the hash matches ALS. The validation split is created exclusively from `trainingRatings`; held-out test rows are never part of validation training, candidate generation, or selection.

In [4]:
val ratingSchema = StructType(Seq(
  StructField("userId", IntegerType, false),
  StructField("movieId", IntegerType, false),
  StructField("rating", FloatType, false),
  StructField("timestamp", LongType, false)
))

val ratingsDF = spark.read
  .option("sep", "::")
  .schema(ratingSchema)
  .csv(ratingsPath)

val ratingCount = ratingsDF.count()
assert(ratingCount == 1000209L, s"Expected 1,000,209 ratings, found $ratingCount")

val splitRanked = ratingsDF
  .withColumn("userRatingCount", count("*").over(Window.partitionBy("userId")))
  .withColumn("shuffleKey", xxhash64(lit(splitSeed), col("userId"), col("movieId")))
  .withColumn("ratingPosition", row_number().over(
    Window.partitionBy("userId").orderBy(col("shuffleKey"), col("movieId"))))
  .withColumn("trainingCount", floor(col("userRatingCount") * trainingFraction))

val trainingRatings = splitRanked.filter(col("ratingPosition") <= col("trainingCount"))
  .select("userId", "movieId", "rating", "timestamp").cache()
val testRatings = splitRanked.filter(col("ratingPosition") > col("trainingCount"))
  .select("userId", "movieId", "rating", "timestamp").cache()

val trainingRatingCount = trainingRatings.count()
val testRatingCount = testRatings.count()
assert(trainingRatingCount + testRatingCount == ratingCount)
println(s"Training: $trainingRatingCount ratings, test: $testRatingCount ratings")

Training: 797758 ratings, test: 202451 ratings


ratingSchema: StructType = StructType(
  StructField("userId", IntegerType, false, {}),
  StructField("movieId", IntegerType, false, {}),
  StructField("rating", FloatType, false, {}),
  StructField("timestamp", LongType, false, {})
)
ratingsDF: DataFrame = [userId: int, movieId: int ... 2 more fields]
ratingCount: Long = 1000209L
splitRanked: DataFrame = [userId: int, movieId: int ... 6 more fields]
trainingRatings: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
testRatings: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
trainingRatingCount: Long = 797758L
testRatingCount: Long = 202451L

## 3. Load the baseline exports and verify they match this split

The exported candidates must contain no training pair, and the top-10 baseline must be exactly the first ten candidates. A mismatch means the split rebuilt here differs from the one used for ALS.

In [5]:
val candidates = spark.read.parquet(new java.io.File(baselineRunDir, "top-100-candidates").getPath)
  .select("userId", "movieId", "recommendationRank", "relevanceScore")
val baselineTop10 = spark.read.parquet(new java.io.File(baselineRunDir, "top-10-baseline").getPath)
  .select("userId", "movieId", "recommendationRank", "relevanceScore")

val trainingUsers = trainingRatings.select("userId").distinct()
val nUsers = trainingUsers.count()

val candidatesInTraining = candidates.join(
  trainingRatings.select("userId", "movieId"), Seq("userId", "movieId"), "inner").count()
val candidateRows = candidates.count()
val baselineFromCandidates = candidates.filter(col("recommendationRank") <= listSize)
val baselineMismatch = baselineTop10.exceptAll(baselineFromCandidates).count() +
  baselineFromCandidates.exceptAll(baselineTop10).count()

println(s"Users: $nUsers, candidate rows: $candidateRows")
println(s"Candidate pairs found in rebuilt training split: $candidatesInTraining")
println(s"Baseline vs first-10 candidate mismatches: $baselineMismatch")
assert(candidatesInTraining == 0L, "Split mismatch: candidates overlap the rebuilt training set.")
assert(candidateRows == nUsers * candidatePoolSize, "Every user must have 100 candidates.")
assert(baselineMismatch == 0L, "Exported baseline must equal the first 10 candidates.")

Users: 6040, candidate rows: 604000
Candidate pairs found in rebuilt training split: 0
Baseline vs first-10 candidate mismatches: 0


candidates: DataFrame = [userId: int, movieId: int ... 2 more fields]
baselineTop10: DataFrame = [userId: int, movieId: int ... 2 more fields]
trainingUsers: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int]
nUsers: Long = 6040L
candidatesInTraining: Long = 0L
candidateRows: Long = 604000L
baselineFromCandidates: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
baselineMismatch: Long = 0L

## 4. Build full-training inputs for selected test evaluation

These full outer-training tables are used only after validation selection, for final selected-setting test evaluation. Validation independently recomputes labels, profiles, segments, and history from its reduced fit partition.

- Movie popularity groups: Head (`H`) while cumulative share before a movie is under 20%, Tail (`T`) once cumulative share including the movie crosses 80%, otherwise Mid (`M`).
- User profiles: shares of observed training rating values in H, M, T.
- Segments: thirds by Head share (`blockbuster-focused`, `diverse`, `niche-focused`).
- Full-training test history buckets: short `<50`, medium `50-200`, long `>200`.

In [6]:
val headShare = 0.20
val tailShare = 0.20
val totalTrainingRatings = trainingRatingCount.toDouble
val movieCounts = trainingRatings.groupBy("movieId").agg(count("*").as("ratingCount"))

val popularityOrder = Window
  .orderBy(col("ratingCount").desc, col("movieId").asc)
  .rowsBetween(Window.unboundedPreceding, Window.currentRow)

val generatedMovieGroups = movieCounts
  .withColumn("cumulativeShare",
    sum(col("ratingCount")).over(popularityOrder) / lit(totalTrainingRatings))
  .withColumn("shareBefore",
    col("cumulativeShare") - col("ratingCount") / lit(totalTrainingRatings))
  .withColumn("popularityGroup",
    when(col("shareBefore") < headShare, "H")
      .when(col("cumulativeShare") > 1.0 - tailShare, "T")
      .otherwise("M"))
  .select("movieId", "ratingCount", "cumulativeShare", "popularityGroup")
  .cache()

val preparedMovieGroups = spark.read.parquet(
  new java.io.File(profileRunDir, "movie-popularity-groups").getPath)
  .select("movieId", "popularityGroup")
val profileConfig = new String(java.nio.file.Files.readAllBytes(
  new java.io.File(profileRunDir, "split-config.txt").toPath), "UTF-8")
  .split("\\r?\\n").flatMap { line =>
    val separator = line.indexOf('=')
    if (separator > 0) Some(line.substring(0, separator).trim -> line.substring(separator + 1).trim) else None
  }.toMap
val expectedProfileConfig = Map(
  "baselineRunId" -> baselineRunId,
  "splitSeed" -> splitSeed.toString,
  "trainingFraction" -> trainingFraction.toString,
  "trainingRatings" -> trainingRatingCount.toString)
require(expectedProfileConfig.forall { case (key, value) => profileConfig.get(key).contains(value) },
  "Popularity profile export was generated from a different training split.")
val generatedMovieLabels = generatedMovieGroups.select("movieId", "popularityGroup")
val movieLabelMismatch = preparedMovieGroups.exceptAll(generatedMovieLabels).count() +
  generatedMovieLabels.exceptAll(preparedMovieGroups).count()
assert(movieLabelMismatch == 0L,
  s"Prepared and regenerated movie labels differ in $movieLabelMismatch rows.")
val movieGroupsOut = preparedMovieGroups.cache()
val movieGroups = generatedMovieGroups

println(s"Exact movie-label comparison passed for ${movieGroupsOut.count()} movies.")
movieGroups.groupBy("popularityGroup")
  .agg(count("*").as("movies"), (sum("ratingCount") / lit(totalTrainingRatings)).as("ratingShare"))
  .orderBy("popularityGroup").show()

Exact movie-label comparison passed for 3685 movies.
+---------------+------+-------------------+
|popularityGroup|movies|        ratingShare|
+---------------+------+-------------------+
|              H|   114|0.20112615605233666|
|              M|  1066| 0.5987116393693326|
|              T|  2505|0.20016220457833078|
+---------------+------+-------------------+



headShare: Double = 0.2
tailShare: Double = 0.2
totalTrainingRatings: Double = 797758.0
movieCounts: DataFrame = [movieId: int, ratingCount: bigint]
popularityOrder: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@842b419
generatedMovieGroups: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [movieId: int, ratingCount: bigint ... 2 more fields]
preparedMovieGroups: DataFrame = [movieId: int, popularityGroup: string]
profileConfig: String = """baselineRunId=2026-11-16T01-44-41.706431Z
splitSeed=42
trainingFraction=0.8
trainingRatings=797758
testRatings=202451
headShare=0.2
tailShare=0.2
historyBuckets=short<50;medium=50-200;long>200
segments=ntile(3) by head share descending, ties by userId
"""
generatedMovieLabels: DataFrame = [movieId: int, popularityGroup: string]
movieLabelMismatch: Long = 0L
movieGroupsOut: DataFrame = [movieId: int, popularityGroup: string]
movieGroups: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [movi

In [7]:
val ratingValue = col("rating").cast("double")
val generatedProfiles = trainingRatings.join(generatedMovieLabels, "movieId")
  .groupBy("userId")
  .agg(
    sum(ratingValue).as("total"),
    sum(when(col("popularityGroup") === "H", ratingValue).otherwise(0.0)).as("h"),
    sum(when(col("popularityGroup") === "M", ratingValue).otherwise(0.0)).as("m"),
    sum(when(col("popularityGroup") === "T", ratingValue).otherwise(0.0)).as("t"))
  .select(col("userId"),
    (col("h") / col("total")).as("head"),
    (col("m") / col("total")).as("mid"),
    (col("t") / col("total")).as("tail"))
  .cache()

val preparedProfiles = spark.read.parquet(
  new java.io.File(profileRunDir, "user-popularity-profiles").getPath)
  .select("userId", "head", "mid", "tail")
val generatedProfileValues = generatedProfiles.select("userId", "head", "mid", "tail")
val profileMismatch = preparedProfiles.exceptAll(generatedProfileValues).count() +
  generatedProfileValues.exceptAll(preparedProfiles).count()
assert(profileMismatch == 0L,
  s"Prepared and regenerated user profiles differ in $profileMismatch rows.")
val profiles = preparedProfiles.cache()
println(s"Exact profile comparison passed for ${profiles.count()} users.")

val segmentTile = ntile(3).over(Window.orderBy(col("head").desc, col("userId").asc))
val segments = profiles.withColumn("tile", segmentTile)
  .select(col("userId"),
    when(col("tile") === 1, "blockbuster-focused")
      .when(col("tile") === 2, "diverse")
      .otherwise("niche-focused").as("segment"))
  .cache()

val userHistory = trainingRatings.groupBy("userId").agg(count("*").as("nTrain"))
  .withColumn("historyBucket",
    when(col("nTrain") < shortCutoff, "short")
      .when(col("nTrain") <= longCutoff, "medium")
      .otherwise("long"))
  .cache()

assert(profiles.count() == nUsers && segments.count() == nUsers && userHistory.count() == nUsers)
println("Segments and history buckets:")
segments.groupBy("segment").count().orderBy("segment").show()
userHistory.groupBy("historyBucket").agg(count("*").as("users"), min("nTrain"), max("nTrain"))
  .orderBy("historyBucket").show()

Exact profile comparison passed for 6040 users.
Segments and history buckets:
+-------------------+-----+
|            segment|count|
+-------------------+-----+
|blockbuster-focused| 2014|
|            diverse| 2013|
|      niche-focused| 2013|
+-------------------+-----+

+-------------+-----+-----------+-----------+
|historyBucket|users|min(nTrain)|max(nTrain)|
+-------------+-----+-----------+-----------+
|         long| 1218|        201|       1851|
|       medium| 2617|         50|        200|
|        short| 2205|         16|         49|
+-------------+-----+-----------+-----------+



ratingValue: org.apache.spark.sql.Column = CAST(rating AS DOUBLE)
generatedProfiles: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, head: double ... 2 more fields]
preparedProfiles: DataFrame = [userId: int, head: double ... 2 more fields]
generatedProfileValues: DataFrame = [userId: int, head: double ... 2 more fields]
profileMismatch: Long = 0L
profiles: DataFrame = [userId: int, head: double ... 2 more fields]
segmentTile: org.apache.spark.sql.Column = ntile(3) OVER (ORDER BY head DESC NULLS LAST, userId ASC NULLS FIRST unspecifiedframe$())
segments: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, segment: string]
userHistory: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, nTrain: bigint ... 1 more field]

## 5. Calibration engine

Everything here runs on the driver with plain Scala arrays, so nothing is serialized to Spark workers. `Calib.js` is base-2 Jensen-Shannon divergence, not square-rooted, so it matches the UPD notebook and ranges over 0-1.

In [8]:
case class Cand(movieId: Int, rank: Int, rel: Double, g: Int)   // g: 0 = H, 1 = M, 2 = T

object Calib {
  private val ln2 = math.log(2.0)

  def js(p: Array[Double], q: Array[Double]): Double = {
    var s = 0.0
    var i = 0
    while (i < 3) {
      val m = (p(i) + q(i)) / 2.0
      if (p(i) > 0.0) s += 0.5 * p(i) * math.log(p(i) / m) / ln2
      if (q(i) > 0.0) s += 0.5 * q(i) * math.log(q(i) / m) / ln2
      i += 1
    }
    s
  }

  def mix(list: Array[Cand]): Array[Double] = {
    val c = Array(0.0, 0.0, 0.0)
    list.foreach(x => c(x.g) += 1.0)
    c.map(_ / list.length)
  }

  // cands must be sorted by ALS rank. Ties go to the better-ranked candidate.
  def rerank(cands: Array[Cand], p: Array[Double], lambda: Double, k: Int): Array[Cand] = {
    val used = new Array[Boolean](cands.length)
    val chosen = new Array[Cand](k)
    val counts = Array(0.0, 0.0, 0.0)
    val q = new Array[Double](3)
    var relSum = 0.0
    var t = 0
    while (t < k) {
      val size = (t + 1).toDouble
      var best = -1
      var bestObj = Double.NegativeInfinity
      var i = 0
      while (i < cands.length) {
        if (!used(i)) {
          val g = cands(i).g
          counts(g) += 1.0
          q(0) = counts(0) / size; q(1) = counts(1) / size; q(2) = counts(2) / size
          counts(g) -= 1.0
          val obj = (1.0 - lambda) * (relSum + cands(i).rel) - lambda * js(p, q)
          if (obj > bestObj) { bestObj = obj; best = i }
        }
        i += 1
      }
      used(best) = true
      chosen(t) = cands(best)
      relSum += cands(best).rel
      counts(cands(best).g) += 1.0
      t += 1
    }
    chosen.sortBy(_.rank)    // present in ALS order; calibration only changes membership
  }
}

// JS sanity checks
val pHead = Array(1.0, 0.0, 0.0); val pTail = Array(0.0, 0.0, 1.0); val pMix = Array(0.2, 0.5, 0.3)
assert(math.abs(Calib.js(pHead, pTail) - 1.0) < 1e-12)
assert(math.abs(Calib.js(pMix, pMix)) < 1e-12)
assert(math.abs(Calib.js(pMix, pHead) - Calib.js(pHead, pMix)) < 1e-12)

val tiedCandidates = Array(Cand(1, 1, 1.0, 0), Cand(2, 2, 1.0, 2))
val tiedBaseline = Calib.rerank(tiedCandidates, pTail, 0.0, 1)
val tiedPositiveWeight = Calib.rerank(tiedCandidates, pTail, 1e-6, 1)
assert(tiedBaseline.map(_.movieId).toSeq == Seq(1), "At lambda=0, tied relevance must keep the ALS-ranked candidate.")
assert(tiedPositiveWeight.map(_.movieId).toSeq == Seq(2),
  "A tiny positive calibration weight may break a relevance tie in favor of the user's popularity profile.")
println("JS and tied-score sanity checks passed.")

JS and tied-score sanity checks passed.


defined class Cand
defined object Calib
pHead: Array[Double] = Array(1.0, 0.0, 0.0)
pTail: Array[Double] = Array(0.0, 0.0, 1.0)
pMix: Array[Double] = Array(0.2, 0.5, 0.3)
tiedCandidates: Array[Cand] = Array(Cand(1, 1, 1.0, 0), Cand(2, 2, 1.0, 2))
tiedBaseline: Array[Cand] = Array(Cand(1, 1, 1.0, 0))
tiedPositiveWeight: Array[Cand] = Array(Cand(2, 2, 1.0, 2))

In [9]:
val groupIndex = Map("H" -> 0, "M" -> 1, "T" -> 2)

val candsByUser: Map[Int, Array[Cand]] = candidates.join(movieGroupsOut, "movieId")
  .select("userId", "movieId", "recommendationRank", "relevanceScore", "popularityGroup")
  .collect()
  .map(r => r.getInt(0) -> Cand(r.getInt(1), r.getInt(2), r.getDouble(3), groupIndex(r.getString(4))))
  .groupBy(_._1)
  .map { case (u, xs) => u -> xs.map(_._2).sortBy(_.rank) }

val profileByUser: Map[Int, Array[Double]] = profiles.collect()
  .map(r => r.getInt(0) -> Array(r.getDouble(1), r.getDouble(2), r.getDouble(3))).toMap
val nTrainByUser: Map[Int, Long] = userHistory.select("userId", "nTrain").collect()
  .map(r => r.getInt(0) -> r.getLong(1)).toMap
val segmentByUser: Map[Int, String] = segments.collect().map(r => r.getInt(0) -> r.getString(1)).toMap
val bucketByUser: Map[Int, String] = userHistory.select("userId", "historyBucket").collect()
  .map(r => r.getInt(0) -> r.getString(1)).toMap

val likedByUser: Map[Int, Set[Int]] = testRatings.filter(col("rating") >= 4.0)
  .select("userId", "movieId").collect()
  .map(r => r.getInt(0) -> r.getInt(1)).groupBy(_._1)
  .map { case (u, xs) => u -> xs.map(_._2).toSet }

assert(candsByUser.size == nUsers.toInt)
assert(candsByUser.values.forall(_.length == candidatePoolSize), "Every candidate movie must have a popularity label.")
println(s"Users with candidates: ${candsByUser.size}; users with a liked test movie: ${likedByUser.size}")

Users with candidates: 6040; users with a liked test movie: 6012


groupIndex: Map[String, Int] = Map("H" -> 0, "M" -> 1, "T" -> 2)
candsByUser: Map[Int, Array[Cand]] = Map(
  2163 -> Array(
    Cand(572, 1, 1.0, 2),
    Cand(318, 2, 0.48849375919138893, 0),
    Cand(1851, 3, 0.4313731557205782, 2),
    Cand(50, 4, 0.429920316103758, 0),
    Cand(3233, 5, 0.40162604228350113, 2),
    Cand(2905, 6, 0.39489069915906017, 2),
    Cand(527, 7, 0.3662823843641026, 0),
    Cand(3245, 8, 0.36542490668572963, 2),
    Cand(2762, 9, 0.3109741849783141, 0),
    Cand(989, 10, 0.3077100082333506, 2),
    Cand(2930, 11, 0.2976180187677715, 2),
    Cand(37, 12, 0.27924928913997943, 2),
    Cand(110, 13, 0.2721839006876537, 0),
    Cand(853, 14, 0.26170425022272725, 2),
    Cand(1704, 15, 0.2428432978214519, 0),
    Cand(593, 16, 0.24190437576798668, 0),
    Cand(1234, 17, 0.23891085041798035, 1),
    Cand(3897, 18, 0.21528984772747076, 1),
    Cand(2329, 19, 0.2150714772195815, 1),
    Cand(1262, 20, 0.2126519746704437, 1),
    Cand(3092, 21, 0.20275488266141725, 2),

## 6. Calibration settings and metric helpers

A `Setting` is one method with its parameters. UPD is computed for all users; relevance metrics use users with at least one liked (rating `>=4`) item in the evaluated holdout split. The same helpers support validation selection and selected-only test evaluation.

In [10]:
case class Setting(method: String, lambda: Double, n0: Option[Int]) {
  def name: String = (method, n0) match {
    case ("baseline", _) => "baseline"
    case ("cp", _)       => f"cp_l$lambda%.1f"
    case (_, Some(n))    => f"ha_n${n}_l$lambda%.1f"
    case _               => method
  }
  def userLambda(n: Long): Double = n0 match {
    case Some(n0v) => lambda * math.min(1.0, n.toDouble / n0v)
    case None      => lambda
  }
}

val historyWeightCheck = Setting("ha", 0.9, Some(100)).userLambda(50)
assert(math.abs(historyWeightCheck - 0.45) < 1e-12,
  s"History-aware weight should be lambda * min(1, n / N0); found $historyWeightCheck.")
assert(math.abs(Setting("ha", 0.9, Some(100)).userLambda(200) - 0.9) < 1e-12,
  "History-aware weight must cap at lambda when n >= N0.")
assert(Setting("ha", 0.9, Some(100)).userLambda(0) == 0.0,
  "History-aware weight must be zero for an empty history.")
println("History-aware weight formula checks passed.")

val settings: Seq[Setting] =
  Seq(Setting("baseline", 0.0, None)) ++
  lambdas.map(l => Setting("cp", l, None)) ++
  (for (n0 <- n0Values; l <- lambdas) yield Setting("ha", l, Some(n0)))

def runSetting(
  s: Setting,
  candidateData: Map[Int, Array[Cand]] = candsByUser,
  profileData: Map[Int, Array[Double]] = profileByUser,
  historyData: Map[Int, Long] = nTrainByUser,
  historyScale: Double = 1.0
): Map[Int, Array[Cand]] =
  candidateData.map { case (u, cands) =>
    val effectiveHistory = math.round(historyData(u).toDouble * historyScale)
    u -> Calib.rerank(cands, profileData(u), s.userLambda(effectiveHistory), listSize)
  }

case class UserRec(userId: Int, segment: String, bucket: String, mismatch: Double,
                   scored: Boolean, precision: Double, recall: Double, ndcg: Double)

def evaluateWith(
  lists: Map[Int, Array[Cand]], profileData: Map[Int, Array[Double]],
  segmentData: Map[Int, String], bucketData: Map[Int, String],
  relevantData: Map[Int, Set[Int]]
): Seq[UserRec] =
  lists.toSeq.map { case (u, list) =>
    val mismatch = Calib.js(profileData(u), Calib.mix(list))
    relevantData.get(u) match {
      case Some(liked) =>
        val hits = list.map(c => liked.contains(c.movieId))
        val nHits = hits.count(identity).toDouble
        val dcg = hits.zipWithIndex.collect {
          case (true, i) => 1.0 / (math.log(i + 2.0) / math.log(2.0))
        }.sum
        val idcg = (0 until math.min(liked.size, listSize))
          .map(i => 1.0 / (math.log(i + 2.0) / math.log(2.0))).sum
        UserRec(u, segmentData(u), bucketData(u), mismatch, true,
          nHits / listSize, nHits / liked.size, dcg / idcg)
      case None => UserRec(u, segmentData(u), bucketData(u), mismatch, false, 0.0, 0.0, 0.0)
    }
  }

def evaluate(lists: Map[Int, Array[Cand]]): Seq[UserRec] =
  evaluateWith(lists, profileByUser, segmentByUser, bucketByUser, likedByUser)

def avgOf(xs: Seq[Double]): Double = if (xs.isEmpty) Double.NaN else xs.sum / xs.size

case class ResultRow(setting: String, method: String, lambda: Double, n0: Int,
                     groupType: String, group: String, users: Int, scoredUsers: Int,
                     upd: Double, precision: Double, recall: Double, ndcg: Double)

def summarize(s: Setting, groupType: String, group: String, recs: Seq[UserRec]): ResultRow = {
  val scored = recs.filter(_.scored)
  ResultRow(s.name, s.method, s.lambda, s.n0.getOrElse(0), groupType, group, recs.size, scored.size,
    avgOf(recs.map(_.mismatch)), avgOf(scored.map(_.precision)), avgOf(scored.map(_.recall)), avgOf(scored.map(_.ndcg)))
}

def officialOverallUPD(recs: Seq[UserRec]): Double =
  avgOf(recs.groupBy(_.segment).values.map(g => avgOf(g.map(_.mismatch))).toSeq)



History-aware weight formula checks passed.


defined class Setting
historyWeightCheck: Double = 0.45
settings: Seq[Setting] = List(
  Setting("baseline", 0.0, None),
  Setting("cp", 0.1, None),
  Setting("cp", 0.3, None),
  Setting("cp", 0.5, None),
  Setting("cp", 0.7, None),
  Setting("cp", 0.9, None),
  Setting("ha", 0.1, Some(50)),
  Setting("ha", 0.3, Some(50)),
  Setting("ha", 0.5, Some(50)),
  Setting("ha", 0.7, Some(50)),
  Setting("ha", 0.9, Some(50)),
  Setting("ha", 0.1, Some(100)),
  Setting("ha", 0.3, Some(100)),
  Setting("ha", 0.5, Some(100)),
  Setting("ha", 0.7, Some(100)),
  Setting("ha", 0.9, Some(100)),
  Setting("ha", 0.1, Some(200)),
  Setting("ha", 0.3, Some(200)),
  Setting("ha", 0.5, Some(200)),
  Setting("ha", 0.7, Some(200)),
  Setting("ha", 0.9, Some(200))
)
defined function runSetting
defined class UserRec
defined function evaluateWith
defined function evaluate
defined function avgOf
defined class ResultRow
defined function summarize
defined function officialOverallUPD

### Validation split and reduced-training profiles

Create a per-user seed-7 80/20 split from `trainingRatings` only. Recompute popularity groups, rating-value profiles, segments, and history from the reduced fit set. Since the fit is smaller, scale original history/N0 cutoffs 50/100/200 to 40/80/160; test-stage cutoffs remain the full-training 50/100/200.

In [11]:
import org.apache.spark.storage.StorageLevel

val (validationTrainFit, validationRatings) = {
  val ranked = trainingRatings
    .withColumn("n", count("*").over(Window.partitionBy("userId")))
    .withColumn("k", xxhash64(lit(validationSplitSeed), col("userId"), col("movieId")))
    .withColumn("pos", row_number().over(Window.partitionBy("userId").orderBy(col("k"), col("movieId"))))
    .withColumn("cut", floor(col("n") * validationTrainingFraction))
  (ranked.filter(col("pos") <= col("cut")).select("userId", "movieId", "rating", "timestamp"),
   ranked.filter(col("pos") > col("cut")).select("userId", "movieId", "rating", "timestamp"))
}
validationTrainFit.persist(StorageLevel.MEMORY_AND_DISK)
validationRatings.persist(StorageLevel.MEMORY_AND_DISK)
val validationTrainCount = validationTrainFit.count()
val validationHoldoutCount = validationRatings.count()
val validationOverlap = validationTrainFit.select("userId", "movieId")
  .join(validationRatings.select("userId", "movieId"), Seq("userId", "movieId"), "inner").limit(1).count()
assert(validationTrainCount > 0L && validationHoldoutCount > 0L)
assert(validationTrainCount + validationHoldoutCount == trainingRatingCount,
  "Validation split must partition trainingRatings exactly.")
assert(validationOverlap == 0L, "Validation fit and holdout overlap.")
println(s"PASS: training-only validation split; fit=$validationTrainCount validation=$validationHoldoutCount overlap=$validationOverlap")

val validationMovieCounts = validationTrainFit.groupBy("movieId").agg(count("*").as("ratingCount"))
val validationTotalRatings = validationTrainCount.toDouble
val validationOrder = Window.orderBy(col("ratingCount").desc, col("movieId").asc)
  .rowsBetween(Window.unboundedPreceding, Window.currentRow)
val validationMovieGroups = validationMovieCounts
  .withColumn("cumulativeShare", sum(col("ratingCount")).over(validationOrder) / lit(validationTotalRatings))
  .withColumn("shareBefore", col("cumulativeShare") - col("ratingCount") / lit(validationTotalRatings))
  .withColumn("popularityGroup", when(col("shareBefore") < headShare, "H")
    .when(col("cumulativeShare") > 1.0 - tailShare, "T").otherwise("M"))
  .select("movieId", "popularityGroup").cache()
val validationRatingValue = col("rating").cast("double")
val validationProfiles = validationTrainFit.join(validationMovieGroups, "movieId").groupBy("userId")
  .agg(sum(validationRatingValue).as("total"),
    sum(when(col("popularityGroup") === "H", validationRatingValue).otherwise(0.0)).as("h"),
    sum(when(col("popularityGroup") === "M", validationRatingValue).otherwise(0.0)).as("m"),
    sum(when(col("popularityGroup") === "T", validationRatingValue).otherwise(0.0)).as("t"))
  .select(col("userId"), (col("h") / col("total")).as("head"),
    (col("m") / col("total")).as("mid"), (col("t") / col("total")).as("tail")).cache()
val validationUsers = validationTrainFit.select("userId").distinct()
val validationUserCount = validationUsers.count()
val validationTile = ntile(3).over(Window.orderBy(col("head").desc, col("userId").asc))
val validationSegments = validationProfiles.withColumn("tile", validationTile)
  .select(col("userId"), when(col("tile") === 1, "blockbuster-focused")
    .when(col("tile") === 2, "diverse").otherwise("niche-focused").as("segment")).cache()
val validationUserHistory = validationTrainFit.groupBy("userId").agg(count("*").as("nTrain"))
  .withColumn("historyBucket", when(col("nTrain") < validationShortCutoff, "short")
    .when(col("nTrain") <= validationLongCutoff, "medium").otherwise("long")).cache()
assert(validationProfiles.count() == validationUserCount && validationSegments.count() == validationUserCount &&
  validationUserHistory.count() == validationUserCount)
val validationProfileByUser: Map[Int, Array[Double]] = validationProfiles.collect()
  .map(r => r.getInt(0) -> Array(r.getDouble(1), r.getDouble(2), r.getDouble(3))).toMap
val validationNByUser: Map[Int, Long] = validationUserHistory.select("userId", "nTrain").collect()
  .map(r => r.getInt(0) -> r.getLong(1)).toMap
val validationSegmentByUser: Map[Int, String] = validationSegments.collect()
  .map(r => r.getInt(0) -> r.getString(1)).toMap
val validationBucketByUser: Map[Int, String] = validationUserHistory.select("userId", "historyBucket")
  .collect().map(r => r.getInt(0) -> r.getString(1)).toMap
val validationLikedByUser: Map[Int, Set[Int]] = validationRatings.filter(col("rating") >= 4.0)
  .select("userId", "movieId").collect().map(r => r.getInt(0) -> r.getInt(1)).groupBy(_._1)
  .map { case (u, xs) => u -> xs.map(_._2).toSet }
assert(validationLikedByUser.nonEmpty)
println(s"PASS: recomputed reduced-training labels=${validationMovieGroups.count()}, profiles=$validationUserCount, relevant users=${validationLikedByUser.size}")
println(s"Validation thresholds: history buckets <$validationShortCutoff / <=$validationLongCutoff; scaled N0 values ${n0Values.map(n => math.round(n * validationTrainingFraction)).mkString(",")}")


PASS: training-only validation split; fit=635723 validation=162035 overlap=0
PASS: recomputed reduced-training labels=3654, profiles=6040, relevant users=6007
Validation thresholds: history buckets <40 / <=160; scaled N0 values 40,80,160


import org.apache.spark.storage.StorageLevel
validationTrainFit: DataFrame = [userId: int, movieId: int ... 2 more fields]
validationRatings: DataFrame = [userId: int, movieId: int ... 2 more fields]
res11_2: DataFrame = [userId: int, movieId: int ... 2 more fields]
res11_3: DataFrame = [userId: int, movieId: int ... 2 more fields]
validationTrainCount: Long = 635723L
validationHoldoutCount: Long = 162035L
validationOverlap: Long = 0L
validationMovieCounts: DataFrame = [movieId: int, ratingCount: bigint]
validationTotalRatings: Double = 635723.0
validationOrder: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@157c2e95
validationMovieGroups: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [movieId: int, popularityGroup: string]
validationRatingValue: org.apache.spark.sql.Column = CAST(rating AS DOUBLE)
validationProfiles: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, head: double ... 2 more fields]
validationUse

### Train ALS on the reduced training set and generate validation candidates

Fit rank-50 ALS with regParam 0.1, 10 iterations, seed `alsSeed` (equal to the configured split seed), exclusively on `validationTrainFit`. Generate unseen-pair candidates from this model, excluding only pairs used in that reduced fit. No held-out test rows are used.

In [12]:
import org.apache.spark.ml.recommendation.ALS

val validationAls = new ALS().setUserCol("userId").setItemCol("movieId").setRatingCol("rating")
  .setImplicitPrefs(false).setRank(50).setMaxIter(10).setRegParam(0.1).setSeed(alsSeed)
  .setColdStartStrategy("drop")
val validationModel = validationAls.fit(validationTrainFit)
val validationLearnedUsers = validationModel.userFactors.select(col("id").as("userId"))
val validationLearnedMovies = validationModel.itemFactors.select(col("id").as("movieId"))
assert(validationLearnedUsers.count() == validationUserCount,
  "Validation ALS must learn every reduced-training user.")
val validationCandidatePairs = validationLearnedUsers.crossJoin(validationLearnedMovies)
  .join(validationTrainFit.select("userId", "movieId"), Seq("userId", "movieId"), "left_anti")
val validationScored = validationModel.transform(validationCandidatePairs).persist(StorageLevel.DISK_ONLY)
assert(validationScored.count() == validationModel.userFactors.count() * validationModel.itemFactors.count() - validationTrainCount,
  "Validation model must score every pair eligible under its reduced training data.")
val validationRank = Window.partitionBy("userId").orderBy(col("prediction").desc, col("movieId").asc)
val validationTop = validationScored.withColumn("recommendationRank", row_number().over(validationRank))
  .filter(col("recommendationRank") <= candidatePoolSize)
val validationBounds = validationTop.groupBy("userId")
  .agg(min("prediction").as("minPrediction"), max("prediction").as("maxPrediction"))
val validationScaled = validationTop.join(validationBounds, "userId")
  .withColumn("relevanceScore", when(col("maxPrediction") === col("minPrediction"), lit(1.0))
    .otherwise((col("prediction") - col("minPrediction")) / (col("maxPrediction") - col("minPrediction"))))
  .select("userId", "movieId", "recommendationRank", "relevanceScore").cache()
val invalidValidationPools = validationScaled.groupBy("userId")
  .agg(count("*").as("items"), countDistinct("movieId").as("movies"),
    min("recommendationRank").as("firstRank"), max("recommendationRank").as("lastRank"))
  .filter(col("items") =!= candidatePoolSize || col("movies") =!= candidatePoolSize ||
    col("firstRank") =!= 1 || col("lastRank") =!= candidatePoolSize).count()
assert(invalidValidationPools == 0L, s"Invalid validation candidate pools: $invalidValidationPools")
val validationGroupIndex = Map("H" -> 0, "M" -> 1, "T" -> 2)
val validationCandsByUser: Map[Int, Array[Cand]] = validationScaled.join(validationMovieGroups, "movieId")
  .select("userId", "movieId", "recommendationRank", "relevanceScore", "popularityGroup")
  .collect()
  .map(r => r.getInt(0) -> Cand(r.getInt(1), r.getInt(2), r.getDouble(3), validationGroupIndex(r.getString(4))))
  .groupBy(_._1)
  .map { case (u, xs) => u -> xs.map(_._2).sortBy(_.rank) }
assert(validationCandsByUser.size == validationUserCount &&
  validationCandsByUser.values.forall(_.length == candidatePoolSize))
validationScored.unpersist(blocking = true)
validationScaled.unpersist(blocking = true)
validationModel.userFactors.unpersist(blocking = true)
validationModel.itemFactors.unpersist(blocking = true)
System.gc()
println(s"PASS: rank-50/regParam-0.1 reduced-training ALS generated ${validationCandsByUser.size} x $candidatePoolSize validation candidates")


PASS: rank-50/regParam-0.1 reduced-training ALS generated 6040 x 100 validation candidates


import org.apache.spark.ml.recommendation.ALS
validationAls: ALS = als_6d0c59bb32f6
validationModel: org.apache.spark.ml.recommendation.ALSModel = ALSModel: uid=als_6d0c59bb32f6, rank=50
validationLearnedUsers: DataFrame = [userId: int]
validationLearnedMovies: DataFrame = [movieId: int]
validationCandidatePairs: DataFrame = [userId: int, movieId: int]
validationScored: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 1 more field]
validationRank: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@1a8ae823
validationTop: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
validationBounds: DataFrame = [userId: int, minPrediction: float ... 1 more field]
validationScaled: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
invalidValidationPools: Long = 0L
validationGroupIndex: Map[String, Int] = Map("H" -> 0,

### Evaluate and select on validation

Run the unchanged `Calib` engine for all 21 settings. Per method, select the highest validation NDCG@10 under `validationUpdMax`. For matched-UPD analysis, include lambda-zero anchors, interpolate CP and each HA N0 curve over 50 points in the common UPD range, and compute mean HA-minus-CP NDCG gains as in `analysis/history_group_analysis.py`. The matched-UPD HA selection is restricted to that criterion's best N0 family and also observes the UPD ceiling.

In [13]:
val validationSettingsRows = scala.collection.mutable.ArrayBuffer[ResultRow]()
val validationListsBySetting = scala.collection.mutable.Map[String, Map[Int, Array[Cand]]]()
settings.foreach { setting =>
  val lists = runSetting(setting, validationCandsByUser, validationProfileByUser,
    validationNByUser, historyScale = 1.0 / validationTrainingFraction)
  validationListsBySetting += setting.name -> lists
  val recs = evaluateWith(lists, validationProfileByUser, validationSegmentByUser,
    validationBucketByUser, validationLikedByUser)
  validationSettingsRows += summarize(setting, "all", "all", recs).copy(upd = officialOverallUPD(recs))
  Seq("short", "medium", "long").foreach(b =>
    validationSettingsRows += summarize(setting, "history", b, recs.filter(_.bucket == b)))
  Seq("blockbuster-focused", "diverse", "niche-focused").foreach(g =>
    validationSettingsRows += summarize(setting, "segment", g, recs.filter(_.segment == g)))
  val row = validationSettingsRows.find(r => r.setting == setting.name && r.groupType == "all").get
  println(f"VALIDATION ${setting.name}%-16s UPD=${row.upd}%.4f P@10=${row.precision}%.4f R@10=${row.recall}%.4f NDCG@10=${row.ndcg}%.4f")
}
assert(validationSettingsRows.size == 21 * 7,
  s"Expected 147 validation summary rows, found ${validationSettingsRows.size}")
val validationAllRows = validationSettingsRows.filter(_.groupType == "all")
assert(validationAllRows.size == 21 && validationAllRows.map(_.setting).distinct.size == 21)
val selectionGroups = Seq(("all", "all")) ++ Seq("short", "medium", "long").map(b => ("history", b)) ++
  Seq("blockbuster-focused", "diverse", "niche-focused").map(g => ("segment", g))
def updCurve(rows: Seq[ResultRow]): Seq[(Double, Double)] =
  rows.groupBy(_.upd).values.map(r => (r.head.upd, r.head.ndcg)).toSeq.sortBy(_._1)
def interpolate(curve: Seq[(Double, Double)], x: Double): Double = {
  require(curve.nonEmpty && x >= curve.head._1 - 1e-12 && x <= curve.last._1 + 1e-12)
  if (x <= curve.head._1) curve.head._2 else if (x >= curve.last._1) curve.last._2
  else {
    val hi = curve.indexWhere(_._1 >= x)
    val (x0, y0) = curve(hi - 1); val (x1, y1) = curve(hi)
    if (x1 == x0) y1 else y0 + (y1 - y0) * (x - x0) / (x1 - x0)
  }
}
case class MatchedUpdRow(groupType: String, group: String, n0: Int, overlap: String,
  meanNdcgGain: Double, minGain: Double, maxGain: Double, shareHaBetter: Double)
val matchedUpdRows = for ((gt, g) <- selectionGroups; n0 <- n0Values) yield {
  val rows = validationSettingsRows.filter(r => r.groupType == gt && r.group == g)
  val cp = updCurve(rows.filter(r => r.method == "baseline" || r.method == "cp"))
  val ha = updCurve(rows.filter(r => r.method == "baseline" || (r.method == "ha" && r.n0 == n0)))
  val lo = math.max(cp.head._1, ha.head._1); val hi = math.min(cp.last._1, ha.last._1)
  if (lo >= hi) MatchedUpdRow(gt, g, n0, "none", Double.NaN, Double.NaN, Double.NaN, Double.NaN)
  else {
    val grid = (0 until 50).map(i => lo + (hi - lo) * i / 49.0)
    val gains = grid.map(x => interpolate(ha, x) - interpolate(cp, x))
    MatchedUpdRow(gt, g, n0, f"$lo%.3f-$hi%.3f", gains.sum / gains.size,
      gains.min, gains.max, gains.count(_ > 0).toDouble / gains.size)
  }
}
val matchedN0Means = matchedUpdRows.filter(r => !r.meanNdcgGain.isNaN).groupBy(_.n0)
  .map { case (n0, rows) => n0 -> rows.map(_.meanNdcgGain).sum / rows.size }
require(matchedN0Means.nonEmpty, "No overlapping CP/HA UPD curves.")
val matchedN0Winner = matchedN0Means.toSeq.sortBy { case (n0, gain) => (-gain, n0) }.head._1
println("Matched-UPD mean NDCG gain by N0 across all/history/segment groups:")
matchedN0Means.toSeq.sortBy(_._1).foreach { case (n0, gain) => println(f"N0=$n0%3d gain=$gain%.6f") }

case class ValidationSelection(criterion: String, setting: Setting, validation: ResultRow, matchedGain: Double)
val primarySelections = validationAllRows.groupBy(_.method).toSeq.map { case (method, rows) =>
  val eligible = rows.filter(_.upd <= validationUpdMax)
  require(eligible.nonEmpty, s"No $method setting meets validation UPD <= $validationUpdMax")
  val winner = eligible.sortBy(r => (-r.ndcg, r.upd, r.setting)).head
  ValidationSelection(s"best-validation-NDCG:$method", settings.find(_.name == winner.setting).get, winner, Double.NaN)
}
val matchedScores = validationAllRows.filter(r => r.method == "ha" && r.n0 == matchedN0Winner && r.upd <= validationUpdMax).flatMap { candidate =>
  val gains = selectionGroups.flatMap { case (gt, g) =>
    val rows = validationSettingsRows.filter(r => r.groupType == gt && r.group == g)
    // Compare this setting's metrics for the same group against that group's CP curve.
    val groupCandidate = rows.find(_.setting == candidate.setting).get
    val cp = updCurve(rows.filter(r => r.method == "baseline" || r.method == "cp"))
    if (groupCandidate.upd >= cp.head._1 && groupCandidate.upd <= cp.last._1)
      Some(groupCandidate.ndcg - interpolate(cp, groupCandidate.upd)) else None
  }
  if (gains.nonEmpty) Some((candidate, gains.sum / gains.size)) else None
}
require(matchedScores.nonEmpty, s"No matched-UPD HA setting with N0=$matchedN0Winner meets the UPD ceiling and CP overlap")
val (matchedWinner, matchedGain) = matchedScores.sortBy { case (r, gain) => (-gain, -r.ndcg, r.upd, r.setting) }.head
val matchedSelection = ValidationSelection("matched-UPD:HA-vs-CP", settings.find(_.name == matchedWinner.setting).get,
  matchedWinner, matchedGain)
val validationSelections = primarySelections :+ matchedSelection
println(s"PASS: selection completed under validation UPD ceiling $validationUpdMax")
validationSelections.foreach(s => println(f"SELECT ${s.criterion}%-28s ${s.setting.name}%-16s UPD=${s.validation.upd}%.4f NDCG=${s.validation.ndcg}%.4f matchedGain=${s.matchedGain}%.6f"))


VALIDATION baseline         UPD=0.2800 P@10=0.0443 R@10=0.0391 NDCG@10=0.0430
VALIDATION cp_l0.1          UPD=0.2605 P@10=0.0447 R@10=0.0394 NDCG@10=0.0433
VALIDATION cp_l0.3          UPD=0.2165 P@10=0.0460 R@10=0.0402 NDCG@10=0.0442
VALIDATION cp_l0.5          UPD=0.1685 P@10=0.0479 R@10=0.0418 NDCG@10=0.0458
VALIDATION cp_l0.7          UPD=0.1099 P@10=0.0519 R@10=0.0450 NDCG@10=0.0492
VALIDATION cp_l0.9          UPD=0.0295 P@10=0.0584 R@10=0.0503 NDCG@10=0.0572
VALIDATION ha_n50_l0.1      UPD=0.2633 P@10=0.0447 R@10=0.0393 NDCG@10=0.0433
VALIDATION ha_n50_l0.3      UPD=0.2264 P@10=0.0458 R@10=0.0398 NDCG@10=0.0440
VALIDATION ha_n50_l0.5      UPD=0.1867 P@10=0.0477 R@10=0.0411 NDCG@10=0.0454
VALIDATION ha_n50_l0.7      UPD=0.1399 P@10=0.0512 R@10=0.0432 NDCG@10=0.0483
VALIDATION ha_n50_l0.9      UPD=0.0786 P@10=0.0570 R@10=0.0465 NDCG@10=0.0546
VALIDATION ha_n100_l0.1     UPD=0.2659 P@10=0.0446 R@10=0.0392 NDCG@10=0.0432
VALIDATION ha_n100_l0.3     UPD=0.2369 P@10=0.0458 R@10=0.0398 N

validationSettingsRows: collection.mutable.ArrayBuffer[ResultRow] = ArrayBuffer(
  ResultRow(
    "baseline",
    "baseline",
    0.0,
    0,
    "all",
    "all",
    6040,
    6007,
    0.27999541271596984,
    0.044281671383385054,
    0.03908447650936644,
    0.04297520421190115
  ),
  ResultRow(
    "baseline",
    "baseline",
    0.0,
    0,
    "history",
    "short",
    2205,
    2173,
    0.28754555734189996,
    0.021214910262310326,
    0.04950803146845493,
    0.031010769115405082
  ),
  ResultRow(
    "baseline",
    "baseline",
    0.0,
    0,
    "history",
    "medium",
    2620,
    2619,
    0.2776282157773443,
...
validationListsBySetting: collection.mutable.Map[String, Map[Int, Array[Cand]]] = Map(
  "ha_n50_l0.1" -> Map(
    2163 -> Array(
      Cand(572, 1, 1.0, 2),
      Cand(318, 2, 0.4622297505966029, 0),
      Cand(527, 3, 0.446374245472837, 0),
      Cand(3245, 4, 0.408158719760423, 2),
      Cand(3338, 5, 0.40655954330634975, 2),
      Cand(50, 6, 0.3974383

### Popularity and exposure diagnostics

Compute average recommended popularity (the per-user mean of the ten recommended movies' full-training `ratingCount`, averaged across users) and aggregate exposure diagnostics for the same 21 reduced-training validation recommendation lists. Report ARP, distinct items, catalog coverage, and item-exposure Gini overall and by validation segment/history bucket. H/M/T exposure shares use the full outer-training movie-popularity labels and are printed per setting. The separate `metrics-popularity-diversity.csv` does not change `results.csv`; these diagnostics use no held-out test ratings. Run `python analysis/popularity_diversity_analysis.py outputs/calibration/<run>` to plot overall validation ARP and coverage against validation UPD.

In [14]:
case class PopularityDiversityRow(setting:String, method:String, lambda:Double, n0:Int,
  groupType:String, group:String, arp:Double, distinctItems:Int, coverage:Double, gini:Double)
val trainedMovieRows = movieGroups.select("movieId", "ratingCount", "popularityGroup").collect()
val trainedMovieCount = trainedMovieRows.length
// The trained catalog depends on the split (seed 42: 3,685 movies; 43: 3,687; 44: 3,664), so derive the
// expected size from this run's training ratings rather than hard-coding one seed's value.
val expectedTrainedMovieCount = trainingRatings.select("movieId").distinct().count()
require(trainedMovieCount.toLong == expectedTrainedMovieCount,
  s"Expected $expectedTrainedMovieCount trained movies in the seed-$splitSeed training split, found $trainedMovieCount")
val popularityGroupIndex = Map("H" -> 0, "M" -> 1, "T" -> 2)
val popularityByMovie: Map[Int, (Long, String)] = trainedMovieRows.map(r =>
  r.getInt(0) -> (r.getLong(1), r.getString(2))).toMap
def exposureGini(counts: Iterable[Long], itemCount: Int): Double = {
  val sorted = counts.toSeq.sorted
  val total = sorted.sum.toDouble
  if (total == 0.0) 0.0 else sorted.zipWithIndex.map { case (x, i) =>
    (2.0 * (i + 1) - itemCount - 1) * x
  }.sum / (itemCount.toDouble * total)
}
val popularityDiversityRows = settings.flatMap { setting =>
  val lists = validationListsBySetting(setting.name)
  assert(lists.size == validationUserCount.toInt &&
    lists.values.forall(list => list.length == listSize && list.map(_.movieId).distinct.length == listSize),
    s"Every ${setting.name} user must have exactly $listSize distinct recommendations.")
  val hmtCounts = Array(0L, 0L, 0L)
  val exposureCounts = scala.collection.mutable.Map[Int, Long]().withDefaultValue(0L)
  lists.values.flatten.foreach { candidate =>
    val (_, popularityGroup) = popularityByMovie.getOrElse(candidate.movieId,
      throw new IllegalArgumentException(s"Recommended movie ${candidate.movieId} is outside the trained catalog."))
    val groupIndex = popularityGroupIndex(popularityGroup)
    hmtCounts(groupIndex) += 1L
    exposureCounts(candidate.movieId) += 1L
  }
  val totalRecommendations = lists.size * listSize
  val hmtShares = hmtCounts.map(_.toDouble / totalRecommendations.toDouble)
  assert(math.abs(hmtShares.sum - 1.0) < 1e-12,
    s"H/M/T recommendation shares must sum to 1 for ${setting.name}.")
  println(f"H/M/T shares ${setting.name}%-16s H=${hmtShares(0)}%.6f M=${hmtShares(1)}%.6f T=${hmtShares(2)}%.6f sum=${hmtShares.sum}%.12f")
  val groups = Seq(("all", "all")) ++ Seq("blockbuster-focused", "diverse", "niche-focused").map(g => ("segment", g)) ++
    Seq("short", "medium", "long").map(g => ("history", g))
  groups.map { case (groupType, group) =>
    val users = lists.toSeq.filter { case (u, _) =>
      groupType match {
        case "all" => true
        case "segment" => validationSegmentByUser(u) == group
        case "history" => validationBucketByUser(u) == group
      }
    }
    require(users.nonEmpty, s"No users in $groupType=$group for ${setting.name}.")
    val userMeanPopularity = users.map { case (_, list) =>
      list.map(c => popularityByMovie(c.movieId)._1.toDouble).sum / listSize
    }.sum / users.size
    val groupExposure = scala.collection.mutable.Map[Int, Long]().withDefaultValue(0L)
    users.foreach { case (_, list) => list.foreach(c => groupExposure(c.movieId) += 1L) }
    val distinct = groupExposure.size
    assert(distinct <= trainedMovieCount,
      s"${setting.name} $groupType=$group recommends $distinct items from a $trainedMovieCount-item catalog.")
    PopularityDiversityRow(setting.name, setting.method, setting.lambda, setting.n0.getOrElse(0),
      groupType, group, userMeanPopularity, distinct, distinct.toDouble / trainedMovieCount,
      exposureGini((0 until trainedMovieCount).map(i => groupExposure.getOrElse(trainedMovieRows(i).getInt(0), 0L)), trainedMovieCount))
  }
}
assert(popularityDiversityRows.size == 21 * 7 && popularityDiversityRows.map(_.setting).distinct.size == 21,
  s"Expected 147 popularity/diversity rows for all 21 settings; found ${popularityDiversityRows.size}.")
println(s"PASS: per-user list size is $listSize for all 21 settings; distinct items never exceed $trainedMovieCount trained movies.")
println(s"PASS: H/M/T recommendation shares sum to 1 for all 21 settings.")
println(s"PASS: computed ${popularityDiversityRows.size} overall/segment/history popularity-diversity rows.")


H/M/T shares baseline         H=0.271788 M=0.156192 T=0.572020 sum=1.000000000000
H/M/T shares cp_l0.1          H=0.270629 M=0.168874 T=0.560497 sum=1.000000000000
H/M/T shares cp_l0.3          H=0.268675 M=0.200281 T=0.531043 sum=1.000000000000
H/M/T shares cp_l0.5          H=0.269205 M=0.242947 T=0.487848 sum=1.000000000000
H/M/T shares cp_l0.7          H=0.273212 M=0.309834 T=0.416954 sum=1.000000000000
H/M/T shares cp_l0.9          H=0.286672 M=0.444950 T=0.268377 sum=1.000000000000
H/M/T shares ha_n50_l0.1      H=0.270430 M=0.167219 T=0.562351 sum=1.000000000000
H/M/T shares ha_n50_l0.3      H=0.268344 M=0.194255 T=0.537401 sum=1.000000000000
H/M/T shares ha_n50_l0.5      H=0.267434 M=0.229222 T=0.503344 sum=1.000000000000
H/M/T shares ha_n50_l0.7      H=0.269089 M=0.282384 T=0.448526 sum=1.000000000000
H/M/T shares ha_n50_l0.9      H=0.274321 M=0.386043 T=0.339636 sum=1.000000000000
H/M/T shares ha_n100_l0.1     H=0.270381 M=0.165348 T=0.564272 sum=1.000000000000
H/M/T shares ha_

defined class PopularityDiversityRow
trainedMovieRows: Array[org.apache.spark.sql.Row] = Array(
  [2858,2693,H],
  [1196,2378,H],
  [260,2372,H],
  [1210,2310,H],
  [589,2130,H],
  [2028,2128,H],
  [480,2100,H],
  [1270,2079,H],
  [2571,2078,H],
  [593,2044,H],
  [1580,2033,H],
  [608,1981,H],
  [1198,1972,H],
  [110,1940,H],
  [2762,1924,H],
  [2396,1874,H],
  [1197,1851,H],
  [1617,1842,H],
  [1097,1816,H],
  [318,1803,H],
  [2628,1789,H],
  [527,1788,H],
  [1265,1788,H],
  [858,1787,H],
  [2997,1787,H],
  [356,1767,H],
  [296,1744,H],
  [2716,1741,H],
  [1,1674,H],
  [1240,1637,H],
  [1214,1616,H],
  [2916,1602,H],
  [457,1585,H],
  [3578,1509,H],
  [541,1464,H],
  [50,1450,H],
  [1259,1438,H],
  [1200,1425,H],
...
trainedMovieCount: Int = 3685
popularityGroupIndex: Map[String, Int] = Map("H" -> 0, "M" -> 1, "T" -> 2)
popularityByMovie: Map[Int, (Long, String)] = Map(
  2163 -> (166L, "T"),
  645 -> (14L, "T"),
  892 -> (116L, "T"),
  69 -> (254L, "M"),
  3021 -> (24L, "T"),
  1322 

## 7. Verify the engine before validation selection

1. At `lambda=0`, the engine must return the exported baseline in the same order.
2. Check the history-aware weight formula directly; a tiny positive weight may break relevance ties differently.
3. Evaluate the selected exported baseline against this split's held-out ratings and assert the calibration engine reproduces those metrics. The optional historical regression check applies only to its matching baseline run.

In [15]:
val baselineLists = runSetting(settings.head)
val exportedBaselineByUser: Map[Int, Seq[Int]] = baselineTop10.collect()
  .map(r => (r.getInt(0), r.getInt(2), r.getInt(1))).groupBy(_._1)
  .map { case (u, xs) => u -> xs.sortBy(_._2).map(_._3).toSeq }

val baselineDiffs = baselineLists.count { case (u, l) => l.map(_.movieId).toSeq != exportedBaselineByUser(u) }
assert(baselineDiffs == 0, s"$baselineDiffs users differ between lambda=0 and the exported baseline.")

val exportedBaselineLists: Map[Int, Array[Cand]] = baselineTop10.join(movieGroupsOut, "movieId")
  .select("userId", "movieId", "recommendationRank", "relevanceScore", "popularityGroup")
  .collect()
  .map(r => r.getInt(0) -> Cand(r.getInt(1), r.getInt(2), r.getDouble(3), groupIndex(r.getString(4))))
  .groupBy(_._1)
  .map { case (u, xs) => u -> xs.map(_._2).sortBy(_.rank) }
assert(exportedBaselineLists.size == nUsers.toInt &&
  exportedBaselineLists.values.forall(_.length == listSize),
  "The selected exported baseline must contain exactly one complete top-10 list per training user.")

val exportedBaselineRecs = evaluate(exportedBaselineLists).filter(_.scored)
val calibrationBaselineRecs = evaluate(baselineLists).filter(_.scored)
val exportedPrecision = avgOf(exportedBaselineRecs.map(_.precision))
val exportedRecall = avgOf(exportedBaselineRecs.map(_.recall))
val exportedNdcg = avgOf(exportedBaselineRecs.map(_.ndcg))
val bP = avgOf(calibrationBaselineRecs.map(_.precision))
val bR = avgOf(calibrationBaselineRecs.map(_.recall))
val bN = avgOf(calibrationBaselineRecs.map(_.ndcg))
assert(calibrationBaselineRecs.size == exportedBaselineRecs.size &&
  math.abs(bP - exportedPrecision) < 1e-12 &&
  math.abs(bR - exportedRecall) < 1e-12 &&
  math.abs(bN - exportedNdcg) < 1e-12,
  "Lambda=0 accuracy must match the selected exported baseline evaluated on this split's held-out ratings.")
println(f"Selected exported baseline: users=${exportedBaselineRecs.size}  P@10=$exportedPrecision%.4f  R@10=$exportedRecall%.4f  NDCG@10=$exportedNdcg%.4f")
println(f"Calibration lambda=0:       users=${calibrationBaselineRecs.size}  P@10=$bP%.4f  R@10=$bR%.4f  NDCG@10=$bN%.4f")

if (checkHistoricalBaselineRegression) {
  require(baselineRunId == historicalRegressionBaselineRunId,
    s"Historical accuracy applies only to baseline $historicalRegressionBaselineRunId, not $baselineRunId.")
  assert(math.abs(exportedPrecision - historicalPrecision) < 5e-4 &&
    math.abs(exportedRecall - historicalRecall) < 5e-4 &&
    math.abs(exportedNdcg - historicalNdcg) < 5e-4,
    "Selected baseline differs from its historical regression metrics.")
  println("Historical baseline regression check passed.")
}
println("Engine checks passed.")

Selected exported baseline: users=6012  P@10=0.0567  R@10=0.0391  NDCG@10=0.0513
Calibration lambda=0:       users=6012  P@10=0.0567  R@10=0.0391  NDCG@10=0.0513
Engine checks passed.


baselineLists: Map[Int, Array[Cand]] = Map(
  2163 -> Array(
    Cand(572, 1, 1.0, 2),
    Cand(318, 2, 0.48849375919138893, 0),
    Cand(1851, 3, 0.4313731557205782, 2),
    Cand(50, 4, 0.429920316103758, 0),
    Cand(3233, 5, 0.40162604228350113, 2),
    Cand(2905, 6, 0.39489069915906017, 2),
    Cand(527, 7, 0.3662823843641026, 0),
    Cand(3245, 8, 0.36542490668572963, 2),
    Cand(2762, 9, 0.3109741849783141, 0),
    Cand(989, 10, 0.3077100082333506, 2)
  ),
  645 -> Array(
    Cand(572, 1, 1.0, 2),
    Cand(3245, 2, 0.8655015067782819, 2),
    Cand(858, 3, 0.6396412152879071, 0),
    Cand(2905, 4, 0.6234728113797798, 2),
    Cand(2019, 5, 0.5438701007502122, 1),
    Cand(912, 6, 0.5073993027350195, 0),
    Cand(318, 7, 0.5036943566353924, 0),
    Cand(527, 8, 0.4980921884596702, 0),
    Cand(50, 9, 0.49048007165309737, 0),
    Cand(2309, 10, 0.4871338009549647, 2)
  ),
  892 -> Array(
    Cand(572, 1, 1.0, 2),
    Cand(3233, 2, 0.4540489847471079, 2),
    Cand(527, 3, 0.412381229

## 8. Select settings on validation only

Build a per-user seed-7 80/20 split from `trainingRatings`. Recompute H/M/T labels, rating-value profiles, segments, and history counts from the reduced fit partition. Train rank-50 ALS (regParam 0.1, 10 iterations, seed `alsSeed`) on that partition and generate its candidates. Evaluate all 21 settings against validation items rated at least 4. Select per method by highest validation NDCG@10 under `validationUpdMax`; also run the matched-UPD CP-versus-HA interpolation used by `analysis/history_group_analysis.py`.

In [16]:
case class SelectedTestRow(selectionCriteria:String, setting:String, method:String, lambda:Double, n0:Int,
  groupType:String, group:String, validationUPD:Double, validationNDCG:Double,
  testUsers:Int, testScoredUsers:Int, testUPD:Double, testPrecision:Double, testRecall:Double, testNDCG:Double)
val selectedByName = validationSelections.groupBy(_.setting.name).toSeq.sortBy(_._1).map { case(name,selections) =>
  (selections.head.setting,selections.map(_.criterion).distinct.sorted.mkString(";")) }
assert(selectedByName.nonEmpty)
require(calibrationDir.mkdirs(),
  s"Could not create unique calibration output folder (it may already exist): ${calibrationDir.getPath}")
val selectedTestRows = scala.collection.mutable.ArrayBuffer[SelectedTestRow]()
selectedByName.foreach { case(setting,criteria) =>
  val lists = runSetting(setting)
  assert(lists.values.forall(l => l.length == listSize && l.map(_.movieId).distinct.length == listSize))
  val recs = evaluate(lists)
  val groups = Seq(("all","all")) ++ Seq("short","medium","long").map(b=>("history",b)) ++
    Seq("blockbuster-focused","diverse","niche-focused").map(g=>("segment",g))
  groups.foreach { case(gt,g) =>
    val relevantRecs = if(gt=="all") recs else if(gt=="history") recs.filter(_.bucket==g) else recs.filter(_.segment==g)
    val testRow = if(gt=="all") summarize(setting,gt,g,recs).copy(upd=officialOverallUPD(recs)) else summarize(setting,gt,g,relevantRecs)
    val validationRow = validationSettingsRows.find(r=>r.setting==setting.name && r.groupType==gt && r.group==g).get
    selectedTestRows += SelectedTestRow(criteria,setting.name,setting.method,setting.lambda,setting.n0.getOrElse(0),gt,g,
      validationRow.upd,validationRow.ndcg,testRow.users,testRow.scoredUsers,testRow.upd,testRow.precision,testRow.recall,testRow.ndcg)
  }
  lists.toSeq.flatMap { case(u,list)=>list.zipWithIndex.map { case(c,i)=>(u,c.movieId,i+1,c.rel) } }
    .toDF("userId","movieId","recommendationRank","relevanceScore").coalesce(1).write.mode("errorifexists")
    .parquet(new java.io.File(calibrationDir,s"top-10-${setting.name}").getPath)
  val overall=selectedTestRows.find(r=>r.setting==setting.name && r.groupType=="all").get
  println(f"TEST selected ${setting.name}%-16s by $criteria%-52s UPD=${overall.testUPD}%.4f P@10=${overall.testPrecision}%.4f R@10=${overall.testRecall}%.4f NDCG@10=${overall.testNDCG}%.4f")
}
assert(selectedTestRows.size==selectedByName.size*7,
  s"Expected seven test aggregate rows per selected setting, found ${selectedTestRows.size}")
println(s"PASS: evaluated only ${selectedByName.size} distinct validation-selected settings on test")


TEST selected baseline         by best-validation-NDCG:baseline                        UPD=0.3265 P@10=0.0567 R@10=0.0391 NDCG@10=0.0513
TEST selected cp_l0.9          by best-validation-NDCG:cp                              UPD=0.0314 P@10=0.0843 R@10=0.0549 NDCG@10=0.0777
TEST selected ha_n100_l0.9     by matched-UPD:HA-vs-CP                                 UPD=0.1374 P@10=0.0787 R@10=0.0476 NDCG@10=0.0704
TEST selected ha_n50_l0.9      by best-validation-NDCG:ha                              UPD=0.0836 P@10=0.0824 R@10=0.0514 NDCG@10=0.0746
PASS: evaluated only 4 distinct validation-selected settings on test


defined class SelectedTestRow
selectedByName: Seq[(Setting, String)] = Vector(
  (Setting("baseline", 0.0, None), "best-validation-NDCG:baseline"),
  (Setting("cp", 0.9, None), "best-validation-NDCG:cp"),
  (Setting("ha", 0.9, Some(100)), "matched-UPD:HA-vs-CP"),
  (Setting("ha", 0.9, Some(50)), "best-validation-NDCG:ha")
)
selectedTestRows: collection.mutable.ArrayBuffer[SelectedTestRow] = ArrayBuffer(
  SelectedTestRow(
    "best-validation-NDCG:baseline",
    "baseline",
    "baseline",
    0.0,
    0,
    "all",
    "all",
    0.27999541271596984,
    0.04297520421190115,
    6040,
    6012,
    0.32653294999916543,
    0.05666999334664026,
    0.03906158255710513,
    0.05125402403241676
  ),
  SelectedTestRow(
    "best-validation-NDCG:baseline",
    "baseline",
    "baseline",
    0.0,
    0,
    "history",
    "short",
    0.28754555734189996,
    0.031010769115405082,
    2205,
    2178,
    0.31843328700597573,
    0.024977043158861515,
    0.04730737836798452,
    0.03191739

## 9. Evaluate only validation-selected settings on test

Use the full-training baseline candidates/profiles only for settings selected on validation. Save the complete validation grid to `results.csv` for selection analysis and the selected-only test aggregates to `selected_on_validation.csv`. The latter labels validation and test metrics separately and records each selection reason.

In [17]:
val manifestText=Seq(
  s"calibrationRunId=$calibrationRunId",s"baselineRunId=$baselineRunId",s"profileRunId=$profileRunId",
  s"splitSeed=$splitSeed",s"alsSeed=$alsSeed",s"trainingFraction=$trainingFraction",s"trainingRatings=$trainingRatingCount",s"testRatings=$testRatingCount",
  s"validationSplitSeed=$validationSplitSeed",s"validationTrainingFraction=$validationTrainingFraction",
  s"validationTrainRatings=$validationTrainCount",s"validationRatings=$validationHoldoutCount",s"validationUPDMax=$validationUpdMax",
  s"validationShortCutoff=$validationShortCutoff",s"validationLongCutoff=$validationLongCutoff",
  s"validationN0Values=${n0Values.map(n=>math.round(n*validationTrainingFraction)).mkString(",")}",
  s"testResultsScope=selected validation settings only",s"selectionMethods=${validationSelections.map(_.criterion).distinct.mkString(",")}",
  s"selectedSettings=${selectedByName.map(_._1.name).mkString(",")}",
  s"matchedUpdN0Means=${matchedN0Means.toSeq.sortBy(_._1).map{case(n,g)=>s"$n:$g"}.mkString(",")}",
  s"headShare=$headShare",s"headBoundary=shareBefore < headShare",s"tailShare=$tailShare",
  s"tailBoundary=cumulativeShare > 1.0 - tailShare (includes crossing movie)",
  "movieSort=ratingCount descending, movieId ascending","calibrationMethods=baseline,CP,HA",
  "calibrationObjective=(1 - lambda_u) * Rel(L) - lambda_u * JS(p(u), q(L))",s"listSize=$listSize",s"candidatePoolSize=$candidatePoolSize",
  s"cpLambdas=${lambdas.mkString(",")}",s"haLambdas=${lambdas.mkString(",")}",s"haN0Values=${n0Values.mkString(",")}",
  s"historicalRegressionCheckEnabled=$checkHistoricalBaselineRegression",s"historicalRegressionBaselineRunId=$historicalRegressionBaselineRunId",
  s"historicalPrecision=$historicalPrecision",s"historicalRecall=$historicalRecall",s"historicalNdcg=$historicalNdcg"
).mkString("\n")+"\n"
val manifestFile=new java.io.File(calibrationDir,"manifest.txt")
java.nio.file.Files.write(manifestFile.toPath,manifestText.getBytes("UTF-8"))
val validationCsvFile=new java.io.File(calibrationDir,"results.csv")
val validationWriter=new java.io.PrintWriter(validationCsvFile)
validationWriter.println("setting,method,lambda,n0,groupType,group,users,scoredUsers,upd,precision,recall,ndcg")
validationSettingsRows.foreach(r=>validationWriter.println(Seq(r.setting,r.method,r.lambda,r.n0,r.groupType,r.group,r.users,r.scoredUsers,r.upd,r.precision,r.recall,r.ndcg).mkString(",")))
validationWriter.close()
val popularityCsv=calibrationDir.toPath.resolve("metrics-popularity-diversity.csv")
val popularityWriter=java.nio.file.Files.newBufferedWriter(popularityCsv,java.nio.charset.StandardCharsets.UTF_8,
  java.nio.file.StandardOpenOption.CREATE_NEW,java.nio.file.StandardOpenOption.WRITE)
try {
  popularityWriter.write("setting,method,lambda,n0,groupType,group,arp,distinctItems,coverage,gini")
  popularityWriter.newLine()
  popularityDiversityRows.foreach(r=>{
    popularityWriter.write(Seq(r.setting,r.method,r.lambda.toString,r.n0.toString,r.groupType,r.group,
      r.arp.toString,r.distinctItems.toString,r.coverage.toString,r.gini.toString).mkString(","))
    popularityWriter.newLine()
  })
} finally popularityWriter.close()
require(java.nio.file.Files.size(popularityCsv)>0L,"Popularity/diversity CSV was not written.")
val selectedCsv=new java.io.File(calibrationDir,"selected_on_validation.csv").toPath
val selectedWriter=java.nio.file.Files.newBufferedWriter(selectedCsv,java.nio.charset.StandardCharsets.UTF_8,
  java.nio.file.StandardOpenOption.CREATE_NEW,java.nio.file.StandardOpenOption.WRITE)
try {
  selectedWriter.write("selectionCriteria,setting,method,lambda,n0,groupType,group,validationUPD,validationNDCG,testUsers,testScoredUsers,testUPD,testPrecision,testRecall,testNDCG")
  selectedWriter.newLine()
  selectedTestRows.foreach(r=>{
    selectedWriter.write(Seq(r.selectionCriteria,r.setting,r.method,r.lambda.toString,r.n0.toString,r.groupType,r.group,
      r.validationUPD.toString,r.validationNDCG.toString,r.testUsers.toString,r.testScoredUsers.toString,
      r.testUPD.toString,r.testPrecision.toString,r.testRecall.toString,r.testNDCG.toString).mkString(","))
    selectedWriter.newLine()
  })
} finally selectedWriter.close()
require(java.nio.file.Files.size(selectedCsv)>0L,"Selected test results CSV was not written.")
val inputsDir=new java.io.File(calibrationDir,"inputs")
profiles.coalesce(1).write.mode("errorifexists").parquet(new java.io.File(inputsDir,"user-popularity-profiles").getPath)
movieGroupsOut.coalesce(1).write.mode("errorifexists").parquet(new java.io.File(inputsDir,"movie-popularity-groups").getPath)
segments.coalesce(1).write.mode("errorifexists").parquet(new java.io.File(inputsDir,"user-segments").getPath)
trainingUsers.coalesce(1).write.mode("errorifexists").parquet(new java.io.File(inputsDir,"evaluation-users").getPath)
userHistory.select("userId","nTrain","historyBucket").coalesce(1).write.mode("errorifexists")
  .parquet(new java.io.File(inputsDir,"user-history").getPath)
println(s"PASS: wrote ${validationSettingsRows.size} validation grid rows to ${validationCsvFile.getPath}")
println(s"PASS: wrote ${popularityDiversityRows.size} popularity/diversity rows to ${popularityCsv.toString}")
println(s"PASS: wrote ${selectedTestRows.size} selected-only test rows to ${selectedCsv.toString}")
println(s"Evaluator inputs in ${inputsDir.getPath}")


PASS: wrote 147 validation grid rows to /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/calibration/2026-11-16T01-57-11.331215Z/results.csv
PASS: wrote 147 popularity/diversity rows to /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/calibration/2026-11-16T01-57-11.331215Z/metrics-popularity-diversity.csv
PASS: wrote 28 selected-only test rows to /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/calibration/2026-11-16T01-57-11.331215Z/selected_on_validation.csv
Evaluator inputs in /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration/outputs/calibration/2026-11-16T01-57-11.331215Z/inputs


manifestText: String = """calibrationRunId=2026-11-16T01-57-11.331215Z
baselineRunId=2026-11-16T01-44-41.706431Z
profileRunId=2026-11-16T01-50-34.019005Z
splitSeed=42
alsSeed=42
trainingFraction=0.8
trainingRatings=797758
testRatings=202451
validationSplitSeed=7
validationTrainingFraction=0.8
validationTrainRatings=635723
validationRatings=162035
validationUPDMax=0.35
validationShortCutoff=40
validationLongCutoff=160
validationN0Values=40,80,160
testResultsScope=selected validation settings only
selectionMethods=best-validation-NDCG:baseline,best-validation-NDCG:cp,best-validation-NDCG:ha,matched-UPD:HA-vs-CP
selectedSettings=baseline,cp_l0.9,ha_n100_l0.9,ha_n50_l0.9
matchedUpdN0Means=50:4.5333995347990953E-4,100:6.970644178805744E-4,200:5.91628910224952E-4
headShare=0.2
headBoundary=shareBefore < headShare
tailShare=0.2
tailBoundary=cumulativeShare > 1.0 - tailShare (includes crossing movie)
movieSort=ratingCount descending, movieId ascending
calibrationMethods=baseline,CP,HA
calibrat

## 10. Cross-check selected test UPD values

Run `baseline-upd-crosscheck.ipynb` with `calibrationRunId` set to this run folder. It compares each selected test list with the corresponding UPD in `selected_on_validation.csv`; the evaluator itself is unchanged.

In [18]:
spark.stop()
println("Spark session stopped.")

Spark session stopped.
